# Day 18: Text Splitters & Chunking Strategies

**Name:** Areeba Amjad  
**Date:** 24 August 2026  
**Topic:** Text Splitters & Chunking Strategies 


### Learning Objectives

- Understand why documents are chunked before embedding and retrieval.
- Compare fixed-size, sentence-based, semantic, and recursive-character chunking.
- Understand chunk size and overlap.
- Practice LangChain text splitters.
- Preserve source, page, and section metadata.
- Understand the trade-off between retrieval precision and context preservation.


## 1. Why Do We Chunk Documents?

Document chunking means dividing a large document into smaller pieces before embedding and retrieval.

### Reasons for Chunking

1. **LLM Context Limits**  
   Large documents may exceed the model's context window. Chunking creates smaller pieces that can be processed efficiently.

2. **Embedding Model Limits**  
   Embedding models have input-size limits. Very long documents may also produce overly broad embeddings.

3. **Retrieval Precision**  
   RAG systems retrieve chunks rather than entire documents. Focused chunks can improve the chance of finding the exact information required.

4. **Better Context Management**  
   A good chunk should contain enough information to be useful without including too much unrelated information.

5. **Efficient Vector Search**  
   Each chunk can have its own embedding and can be independently searched in a vector database.

### Key Point

Chunking determines the units that a retrieval system can return. Poor chunk boundaries can reduce retrieval quality.

## 2. Chunking Strategies

### Fixed-Size Chunking

Text is divided into chunks of a fixed number of characters or tokens.

**Advantages**
- Simple
- Fast
- Predictable

**Disadvantages**
- Can cut sentences or ideas
- Does not understand document structure

### Sentence-Based Chunking

Text is divided at sentence boundaries.

**Advantages**
- Preserves complete sentences
- More natural than arbitrary character splitting

**Disadvantages**
- Sentences can have very different lengths
- A long sentence can still exceed the desired size

### Semantic Chunking

Text is divided according to changes in meaning or topic.

**Advantages**
- Preserves semantic coherence
- Useful for topic-rich documents

**Disadvantages**
- More computationally expensive
- Usually requires semantic similarity calculations
- Chunk sizes can vary

### Recursive Character Chunking

The splitter tries larger separators first and progressively uses smaller separators when necessary.

Typical separators include:
- Paragraph breaks
- Line breaks
- Spaces
- Characters

This is a strong general-purpose approach for RAG.

## 3. Chunk Size and Overlap

A practical starting range for many RAG applications is approximately **256–1024 tokens**.

Possible starting experiments:

- Small: 256–400 tokens
- Medium: 400–800 tokens
- Large: 800–1024 tokens
- Overlap: approximately 10–20%

### Why Overlap?

Overlap repeats some content between neighboring chunks.

This helps when an important sentence or idea crosses a chunk boundary.

### Small Chunks

- More precise retrieval
- Less irrelevant context
- Less surrounding context

### Large Chunks

- More context
- Better continuity
- Can contain more irrelevant information
- Can reduce retrieval precision

There is no universal best chunk size. The best value depends on the documents and questions used by the application.

## 4. LangChain Text Splitters

### RecursiveCharacterTextSplitter

A general-purpose splitter that recursively tries different separators to create appropriately sized chunks.

### TokenTextSplitter

Splits text according to token count. This is useful when token limits are important.

### MarkdownHeaderTextSplitter

Splits Markdown documents according to headings and preserves heading information as metadata.

### Recommended Approach

For generic documents:

`RecursiveCharacterTextSplitter`

For structured Markdown:

`MarkdownHeaderTextSplitter`

followed by recursive splitting if individual sections are too large.

## 5. Research Sources

### AI-Assisted Research

The assignment requires consultation of:

- ChatGPT
- Gemini
- Claude

The research comparison focused on:

- Why chunking is required
- Chunk-size selection
- Chunk overlap
- Retrieval precision
- Semantic chunking
- Metadata preservation
- Small vs large chunk trade-offs

### Articles / Technical Sources

1. LangChain Text Splitters Documentation  
   https://docs.langchain.com/oss/python/integrations/splitters/index

2. LangChain Recursive Text Splitter  
   https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter

3. LangChain Markdown Header Splitter  
   https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter

4. McTaba Labs — Chunking Strategies for RAG  
   https://www.mctaba.com/kb/rag-chunking-strategies

5. Thread Transfer — RAG Document Chunking Best Practices  
   https://thread-transfer.com/blog/2026-06-17-rag-document-chunking-best-practices/

In [1]:
# Install required packages

%pip install -qU langchain-text-splitters tiktoken

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
    MarkdownHeaderTextSplitter,
)

print("LangChain text splitter imports successful!")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


LangChain text splitter imports successful!


## 6. Sample Document

We will use a small AI and RAG document to demonstrate different chunking strategies.

In [3]:
document = """
Artificial intelligence is a field of computer science focused on building
systems that can perform tasks that normally require human intelligence.

Machine learning is a major part of artificial intelligence. Instead of
explicitly programming every rule, machine learning systems learn patterns
from data.

Deep learning is a subset of machine learning that uses neural networks with
multiple layers. It is widely used in computer vision, speech recognition,
natural language processing, and generative AI.

Retrieval-Augmented Generation (RAG) combines information retrieval with
language generation. Documents are usually divided into smaller chunks before
they are converted into embeddings and stored in a vector database.

When a user asks a question, relevant chunks can be retrieved and supplied to
a language model as context. Good chunking can improve retrieval quality and
reduce irrelevant context.
"""

print(document)


Artificial intelligence is a field of computer science focused on building
systems that can perform tasks that normally require human intelligence.

Machine learning is a major part of artificial intelligence. Instead of
explicitly programming every rule, machine learning systems learn patterns
from data.

Deep learning is a subset of machine learning that uses neural networks with
multiple layers. It is widely used in computer vision, speech recognition,
natural language processing, and generative AI.

Retrieval-Augmented Generation (RAG) combines information retrieval with
language generation. Documents are usually divided into smaller chunks before
they are converted into embeddings and stored in a vector database.

When a user asks a question, relevant chunks can be retrieved and supplied to
a language model as context. Good chunking can improve retrieval quality and
reduce irrelevant context.



## 7. Fixed-Size Chunking

Fixed-size chunking divides text into pieces using a predefined character size.

This method is simple but may split sentences or ideas at arbitrary locations.

In [4]:
chunk_size = 300
overlap = 50

fixed_chunks = []
start = 0

while start < len(document):
    end = start + chunk_size
    fixed_chunks.append(document[start:end])
    start += chunk_size - overlap

print("Number of fixed-size chunks:", len(fixed_chunks))

for i, chunk in enumerate(fixed_chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.strip())

Number of fixed-size chunks: 4

--- Chunk 1 ---
Artificial intelligence is a field of computer science focused on building
systems that can perform tasks that normally require human intelligence.

Machine learning is a major part of artificial intelligence. Instead of
explicitly programming every rule, machine learning systems learn patterns
fro

--- Chunk 2 ---
rule, machine learning systems learn patterns
from data.

Deep learning is a subset of machine learning that uses neural networks with
multiple layers. It is widely used in computer vision, speech recognition,
natural language processing, and generative AI.

Retrieval-Augmented Generation (RAG) com

--- Chunk 3 ---
tive AI.

Retrieval-Augmented Generation (RAG) combines information retrieval with
language generation. Documents are usually divided into smaller chunks before
they are converted into embeddings and stored in a vector database.

When a user asks a question, relevant chunks can be retrieved and supp

--- Chunk 4 ---


## 8. RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter` tries to preserve larger text structures before using smaller separators.

It is a useful general-purpose baseline for RAG applications.

In [5]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
)

recursive_chunks = recursive_splitter.create_documents([document])

print("Number of recursive chunks:", len(recursive_chunks))

for i, chunk in enumerate(recursive_chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)

Number of recursive chunks: 5

--- Chunk 1 ---
Artificial intelligence is a field of computer science focused on building
systems that can perform tasks that normally require human intelligence.

--- Chunk 2 ---
Machine learning is a major part of artificial intelligence. Instead of
explicitly programming every rule, machine learning systems learn patterns
from data.

--- Chunk 3 ---
Deep learning is a subset of machine learning that uses neural networks with
multiple layers. It is widely used in computer vision, speech recognition,
natural language processing, and generative AI.

--- Chunk 4 ---
Retrieval-Augmented Generation (RAG) combines information retrieval with
language generation. Documents are usually divided into smaller chunks before
they are converted into embeddings and stored in a vector database.

--- Chunk 5 ---
When a user asks a question, relevant chunks can be retrieved and supplied to
a language model as context. Good chunking can improve retrieval quality and
reduc

## 9. TokenTextSplitter

`TokenTextSplitter` controls chunks according to token count.

Token-based splitting is useful when the application's main constraint is the number of tokens that can be sent to a model.

In [6]:
token_splitter = TokenTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
)

token_chunks = token_splitter.create_documents([document])

print("Number of token-based chunks:", len(token_chunks))

for i, chunk in enumerate(token_chunks[:5], start=1):
    print(f"\n--- Token Chunk {i} ---")
    print(chunk.page_content)

Number of token-based chunks: 2

--- Token Chunk 1 ---

Artificial intelligence is a field of computer science focused on building
systems that can perform tasks that normally require human intelligence.

Machine learning is a major part of artificial intelligence. Instead of
explicitly programming every rule, machine learning systems learn patterns
from data.

Deep learning is a subset of machine learning that uses neural networks with
multiple layers. It is widely used in computer vision, speech recognition,
natural language processing, and generative AI.

Ret

--- Token Chunk 2 ---
 in computer vision, speech recognition,
natural language processing, and generative AI.

Retrieval-Augmented Generation (RAG) combines information retrieval with
language generation. Documents are usually divided into smaller chunks before
they are converted into embeddings and stored in a vector database.

When a user asks a question, relevant chunks can be retrieved and supplied to
a language model as 

## 10. MarkdownHeaderTextSplitter

Markdown headings provide useful document structure.

The splitter can preserve heading information as metadata.

This is particularly useful for:
- Documentation
- README files
- Technical notes
- Knowledge bases

In [7]:
markdown_document = """
# Artificial Intelligence

Artificial intelligence enables computers to perform tasks associated with
human intelligence.

## Machine Learning

Machine learning allows systems to learn patterns from data.

### Supervised Learning

Supervised learning uses labeled examples to train a model.

### Unsupervised Learning

Unsupervised learning discovers patterns in unlabeled data.

## Retrieval-Augmented Generation

RAG retrieves relevant information from a knowledge base and provides it to
a language model as context.

### Chunking

Documents are divided into smaller chunks before embedding and retrieval.

### Embeddings

Embeddings represent text as numerical vectors that can be compared for
semantic similarity.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,
)

markdown_chunks = markdown_splitter.split_text(markdown_document)

print("Number of Markdown chunks:", len(markdown_chunks))

for i, chunk in enumerate(markdown_chunks, start=1):
    print(f"\n--- Markdown Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print("Content:", chunk.page_content)

Number of Markdown chunks: 7

--- Markdown Chunk 1 ---
Metadata: {'Header 1': 'Artificial Intelligence'}
Content: # Artificial Intelligence  
Artificial intelligence enables computers to perform tasks associated with
human intelligence.

--- Markdown Chunk 2 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning'}
Content: ## Machine Learning  
Machine learning allows systems to learn patterns from data.

--- Markdown Chunk 3 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Supervised Learning'}
Content: ### Supervised Learning  
Supervised learning uses labeled examples to train a model.

--- Markdown Chunk 4 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Unsupervised Learning'}
Content: ### Unsupervised Learning  
Unsupervised learning discovers patterns in unlabeled data.

--- Markdown Chunk 5 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'Header

## 11. Metadata Preservation

Metadata tells us where a chunk came from.

Useful metadata includes:

- `source`
- `page`
- `document_id`
- `section`
- `Header 1`
- `Header 2`
- `Header 3`

Metadata is important in RAG because retrieved chunks can be traced back to their original source.

In [9]:
for index, chunk in enumerate(markdown_chunks, start=1):
    chunk.metadata["source"] = "day18_notes.md"
    chunk.metadata["page"] = 1
    chunk.metadata["chunk_id"] = index

for chunk in markdown_chunks:
    print("Metadata:", chunk.metadata)
    print("Content:", chunk.page_content[:150], "...")
    print()

Metadata: {'Header 1': 'Artificial Intelligence', 'source': 'day18_notes.md', 'page': 1, 'chunk_id': 1}
Content: # Artificial Intelligence  
Artificial intelligence enables computers to perform tasks associated with
human intelligence. ...

Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'source': 'day18_notes.md', 'page': 1, 'chunk_id': 2}
Content: ## Machine Learning  
Machine learning allows systems to learn patterns from data. ...

Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Supervised Learning', 'source': 'day18_notes.md', 'page': 1, 'chunk_id': 3}
Content: ### Supervised Learning  
Supervised learning uses labeled examples to train a model. ...

Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Unsupervised Learning', 'source': 'day18_notes.md', 'page': 1, 'chunk_id': 4}
Content: ### Unsupervised Learning  
Unsupervised learning discovers patterns in u

## 12. Combining Markdown Headers with Recursive Chunking

A useful production approach is:

Markdown structure
↓
Preserve section metadata
↓
Recursive chunking
↓
Create final chunks

This provides both structural context and size control.

In [10]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
)

final_chunks = recursive_splitter.split_documents(markdown_chunks)

print("Final number of chunks:", len(final_chunks))

for i, chunk in enumerate(final_chunks, start=1):
    print(f"\n--- Final Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print("Content:", chunk.page_content)

Final number of chunks: 7

--- Final Chunk 1 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'source': 'day18_notes.md', 'page': 1, 'chunk_id': 1}
Content: # Artificial Intelligence  
Artificial intelligence enables computers to perform tasks associated with
human intelligence.

--- Final Chunk 2 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'source': 'day18_notes.md', 'page': 1, 'chunk_id': 2}
Content: ## Machine Learning  
Machine learning allows systems to learn patterns from data.

--- Final Chunk 3 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Supervised Learning', 'source': 'day18_notes.md', 'page': 1, 'chunk_id': 3}
Content: ### Supervised Learning  
Supervised learning uses labeled examples to train a model.

--- Final Chunk 4 ---
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Unsupervised Learning', 'source': 'day18_notes.md', 'page

## 13. Chunk Size Experiment

Different chunk sizes produce different numbers of chunks.

We will compare:

- 150
- 300
- 500
- 800 characters

This experiment demonstrates why chunk size affects the number of retrievable units.

In [11]:
chunk_sizes = [150, 300, 500, 800]

print("Chunk Size | Overlap | Number of Chunks")
print("-" * 42)

for size in chunk_sizes:
    overlap = int(size * 0.10)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
    )

    chunks = splitter.split_text(document)

    print(f"{size:10} | {overlap:7} | {len(chunks):16}")

Chunk Size | Overlap | Number of Chunks
------------------------------------------
       150 |      15 |                9
       300 |      30 |                5
       500 |      50 |                3
       800 |      80 |                2


## 14. Reusable Chunking Pipeline

The following function creates a reusable pipeline that:

1. Splits Markdown according to headings.
2. Preserves heading metadata.
3. Applies recursive chunking.
4. Adds source information.
5. Adds a unique chunk ID.

In [12]:
def build_chunking_pipeline(
    markdown_text,
    source_name,
    chunk_size=300,
    chunk_overlap=50,
):
    headers = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers,
        strip_headers=False,
    )

    section_documents = header_splitter.split_text(markdown_text)

    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunks = recursive_splitter.split_documents(section_documents)

    for index, chunk in enumerate(chunks, start=1):
        chunk.metadata["source"] = source_name
        chunk.metadata["chunk_id"] = index

    return chunks


pipeline_chunks = build_chunking_pipeline(
    markdown_document,
    source_name="day18_notes.md",
    chunk_size=180,
    chunk_overlap=30,
)

print("Created chunks:", len(pipeline_chunks))

for chunk in pipeline_chunks:
    print("=" * 70)
    print("Chunk ID:", chunk.metadata["chunk_id"])
    print("Source:", chunk.metadata["source"])
    print("Metadata:", chunk.metadata)
    print("Content:", chunk.page_content)

Created chunks: 7
Chunk ID: 1
Source: day18_notes.md
Metadata: {'Header 1': 'Artificial Intelligence', 'source': 'day18_notes.md', 'chunk_id': 1}
Content: # Artificial Intelligence  
Artificial intelligence enables computers to perform tasks associated with
human intelligence.
Chunk ID: 2
Source: day18_notes.md
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'source': 'day18_notes.md', 'chunk_id': 2}
Content: ## Machine Learning  
Machine learning allows systems to learn patterns from data.
Chunk ID: 3
Source: day18_notes.md
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Supervised Learning', 'source': 'day18_notes.md', 'chunk_id': 3}
Content: ### Supervised Learning  
Supervised learning uses labeled examples to train a model.
Chunk ID: 4
Source: day18_notes.md
Metadata: {'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Unsupervised Learning', 'source': 'day18_notes.md'

## 15. Chunking Strategy Comparison

| Strategy | Best Use | Main Strength | Main Weakness |
|---|---|---|---|
| Fixed-size | Simple baseline | Easy and predictable | Can break ideas |
| Sentence-based | Natural language | Preserves sentences | Sentence sizes vary |
| Semantic | Topic-rich documents | Preserves meaning | More computationally expensive |
| Recursive character | General RAG | Good balance | Depends on separators |
| Token-based | Token constraints | Direct token control | Tokenizer dependent |
| Markdown header | Documentation | Preserves structure | Requires Markdown |

### Recommended Starting Point

For generic text:

`RecursiveCharacterTextSplitter`

For structured Markdown:

`MarkdownHeaderTextSplitter`

Then use recursive splitting if the sections are too large.

## 16. Small vs Large Chunks

| Feature | Small Chunks | Large Chunks |
|---|---|---|
| Retrieval precision | Usually higher | Can be lower |
| Context per result | Lower | Higher |
| Irrelevant information | Usually lower | Can be higher |
| Number of vectors | Higher | Lower |
| Context preservation | Lower | Higher |
| Storage/indexing cost | Higher | Lower |

### Conclusion

Small chunks are useful when precise retrieval is important.

Large chunks are useful when surrounding context is important.

The best solution is normally found by testing different chunk sizes on the actual RAG dataset.

## 17. Key Findings

1. Chunking makes large documents manageable.
2. Chunking helps with LLM context limitations.
3. Embedding models also have input limitations.
4. Smaller chunks can improve retrieval precision.
5. Larger chunks preserve more context.
6. Overlap helps preserve information across chunk boundaries.
7. `RecursiveCharacterTextSplitter` is a strong general-purpose option.
8. `TokenTextSplitter` is useful for token-based constraints.
9. `MarkdownHeaderTextSplitter` preserves document structure through metadata.
10. Metadata such as source, page, and section should be preserved.
11. There is no universally optimal chunk size.
12. Chunking should be evaluated using the actual documents and queries.

## 18. Final Conclusion

Text chunking is an important preprocessing step in Retrieval-Augmented Generation systems.

A good chunking strategy balances **retrieval precision** and **context preservation**.

Small chunks provide more focused retrieval but may lose surrounding context. Large chunks preserve more context but can contain irrelevant information.

For a first RAG implementation, `RecursiveCharacterTextSplitter` is a practical baseline. For structured Markdown documents, `MarkdownHeaderTextSplitter` can preserve section information, followed by recursive splitting when sections are too large.

### RAG Chunking Pipeline

Document
↓
Structure-aware splitting
↓
Recursive/token chunking
↓
Metadata preservation
↓
Embeddings
↓
Vector database
↓
Similarity retrieval
↓
LLM context
↓
Answer

### References

- LangChain Text Splitters:
  https://docs.langchain.com/oss/python/integrations/splitters/index

- LangChain Recursive Text Splitter:
  https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter

- LangChain Token Splitter:
  https://docs.langchain.com/oss/python/integrations/splitters/split_by_token

- LangChain Markdown Header Splitter:
  https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter

- McTaba Labs — Chunking Strategies for RAG:
  https://www.mctaba.com/kb/rag-chunking-strategies

- Thread Transfer — RAG Document Chunking Best Practices:
  https://thread-transfer.com/blog/2026-06-17-rag-document-chunking-best-practices/

# Task 1: Compare Chunking Methods

In this task, we compare:

1. CharacterTextSplitter
   - Chunk size: 500 characters
   - Overlap: 0

2. RecursiveCharacterTextSplitter
   - Chunk size: 500 characters
   - Overlap: 50

We will compare:
- Number of chunks
- Chunk boundaries
- Context preservation
- Semantic coherence

In [13]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

# Sample long document
long_document = """
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
language understanding, perception, and decision making.

Machine learning is a major branch of artificial intelligence. Machine
learning algorithms learn patterns from data instead of relying entirely on
manually written rules. Common machine learning approaches include supervised
learning, unsupervised learning, and reinforcement learning.

Deep learning is a specialized area of machine learning that uses neural
networks containing multiple layers. Deep learning has achieved strong results
in computer vision, natural language processing, speech recognition, and
generative artificial intelligence.

Natural language processing allows computers to work with human language.
Applications include sentiment analysis, machine translation, text
classification, question answering, information extraction, and chatbots.

Retrieval-Augmented Generation, commonly called RAG, combines information
retrieval with language generation. A RAG system first retrieves relevant
information from a knowledge base and then provides that information to a
language model as context.

Before retrieval, documents are normally divided into smaller chunks.
Each chunk can be converted into an embedding and stored in a vector database.
When a user asks a question, the system searches for chunks that are
semantically related to the query.

Chunk size has a major impact on retrieval quality. Very small chunks may
provide highly precise information but may lose important surrounding
context. Very large chunks preserve more context but may contain irrelevant
information.

Chunk overlap helps preserve information when an important sentence or idea
crosses the boundary between two chunks. However, excessive overlap increases
the number of stored chunks and therefore increases processing and storage
requirements.

Metadata is also important in a retrieval system. Useful metadata includes
the source document, page number, section title, document type, and chunk ID.
This metadata allows retrieved information to be traced back to its original
location.

Good document processing therefore requires a balance between chunk size,
overlap, semantic coherence, retrieval precision, and storage efficiency.
"""

print("Document characters:", len(long_document))

Document characters: 2416


In [14]:
# CharacterTextSplitter
character_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=500,
    chunk_overlap=0,
)

character_chunks = character_splitter.create_documents(
    [long_document]
)

print("CharacterTextSplitter")
print("Number of chunks:", len(character_chunks))

CharacterTextSplitter
Number of chunks: 5


In [15]:
# RecursiveCharacterTextSplitter
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

recursive_chunks = recursive_splitter.create_documents(
    [long_document]
)

print("RecursiveCharacterTextSplitter")
print("Number of chunks:", len(recursive_chunks))

RecursiveCharacterTextSplitter
Number of chunks: 7


In [16]:
print("=" * 80)
print("CHARACTER TEXT SPLITTER")
print("=" * 80)

for i, chunk in enumerate(character_chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)

CHARACTER TEXT SPLITTER

--- Chunk 1 ---
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
language understanding, perception, and decision making.

Machine learning is a major branch of artificial intelligence. Machine
learning algorithms learn patterns from data instead of relying entirely on
manually written rules. Common machine learning approaches include

--- Chunk 2 ---
supervised
learning, unsupervised learning, and reinforcement learning.

Deep learning is a specialized area of machine learning that uses neural
networks containing multiple layers. Deep learning has achieved strong results
in computer vision, natural language processing, speech recognition, and
generative artificial intelligence.

Natural language processing allows computers to work with human language.
Applications include sentiment analysis,

In [17]:
print("=" * 80)
print("RECURSIVE CHARACTER TEXT SPLITTER")
print("=" * 80)

for i, chunk in enumerate(recursive_chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)

RECURSIVE CHARACTER TEXT SPLITTER

--- Chunk 1 ---
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
language understanding, perception, and decision making.

--- Chunk 2 ---
Machine learning is a major branch of artificial intelligence. Machine
learning algorithms learn patterns from data instead of relying entirely on
manually written rules. Common machine learning approaches include supervised
learning, unsupervised learning, and reinforcement learning.

--- Chunk 3 ---
Deep learning is a specialized area of machine learning that uses neural
networks containing multiple layers. Deep learning has achieved strong results
in computer vision, natural language processing, speech recognition, and
generative artificial intelligence.

Natural language processing allows computers to work with human language.
Applications in

In [18]:
# Compare chunk statistics

print("CharacterTextSplitter chunks:", len(character_chunks))
print("RecursiveCharacterTextSplitter chunks:", len(recursive_chunks))

print("\nAverage character length:")

character_avg = sum(
    len(chunk.page_content) for chunk in character_chunks
) / len(character_chunks)

recursive_avg = sum(
    len(chunk.page_content) for chunk in recursive_chunks
) / len(recursive_chunks)

print("Character splitter:", round(character_avg, 2))
print("Recursive splitter:", round(recursive_avg, 2))

CharacterTextSplitter chunks: 5
RecursiveCharacterTextSplitter chunks: 7

Average character length:
Character splitter: 482.0
Recursive splitter: 343.14


## Task 1 Findings

### CharacterTextSplitter

Character-based splitting is simple and predictable. However, fixed boundaries
can separate related sentences or ideas. This can reduce semantic coherence.

### RecursiveCharacterTextSplitter

Recursive splitting attempts to preserve larger text structures before using
smaller separators. The 50-character overlap also provides additional context
between neighboring chunks.

### Result

**RecursiveCharacterTextSplitter preserves meaning and context better than a
simple fixed character splitter.**

The main reason is that recursive splitting tries to respect natural text
boundaries, while overlap reduces information loss at chunk boundaries.

### Recommendation

For general RAG documents, `RecursiveCharacterTextSplitter` is the better
starting point.

# Task 2: Optimal Chunk Size Experiment

We will test four chunk sizes:

- 200 characters
- 500 characters
- 1000 characters
- 2000 characters

For each size we will calculate:

- Number of chunks
- Average chunk length
- Estimated storage
- Retrieval quality
- Recommendation

## Embedding Note

A production RAG system normally uses a neural embedding model.

For this notebook, we avoid requiring PyTorch, an external API, or a paid API.
Instead, we use **TF-IDF vectors as a lightweight embedding/retrieval proxy**.

This is suitable for demonstrating the chunk-size experiment, but TF-IDF is
not equivalent to modern semantic embeddings such as sentence-transformer
embeddings.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

chunk_sizes = [200, 500, 1000, 2000]

questions = [
    "What is retrieval augmented generation?",
    "Why is chunk overlap useful?",
    "What is the purpose of metadata?",
    "What are the problems with very small chunks?",
    "What are the problems with very large chunks?",
]

In [20]:
def create_chunks(text, chunk_size):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size * 0.10),
    )
    
    return splitter.create_documents([text])


chunk_experiment = {}

for size in chunk_sizes:
    chunks = create_chunks(long_document, size)
    chunk_experiment[size] = chunks

    print(
        f"Chunk size {size}: "
        f"{len(chunks)} chunks"
    )

Chunk size 200: 19 chunks
Chunk size 500: 7 chunks
Chunk size 1000: 3 chunks
Chunk size 2000: 2 chunks


In [21]:
# Estimate storage requirements

storage_results = []

for size, chunks in chunk_experiment.items():

    texts = [chunk.page_content for chunk in chunks]

    vectorizer = TfidfVectorizer()

    vectors = vectorizer.fit_transform(texts)

    # Approximate dense float32 storage
    dense_storage_bytes = (
        vectors.shape[0] * vectors.shape[1] * 4
    )

    original_text_bytes = sum(
        len(text.encode("utf-8"))
        for text in texts
    )

    storage_results.append({
        "chunk_size": size,
        "num_chunks": len(chunks),
        "avg_chunk_chars": round(
            np.mean([len(text) for text in texts]), 2
        ),
        "text_storage_bytes": original_text_bytes,
        "embedding_dimensions": vectors.shape[1],
        "estimated_vector_storage_bytes": dense_storage_bytes,
    })

storage_df = pd.DataFrame(storage_results)

storage_df

,chunk_size,num_chunks,avg_chunk_chars,text_storage_bytes,embedding_dimensions,estimated_vector_storage_bytes
0,200,19,125.63,2387,189,14364
1,500,7,343.14,2402,189,5292
2,1000,3,803.33,2410,189,2268
3,2000,2,1206.00,2412,189,1512


In [22]:
# Display storage in KB

storage_df["text_storage_KB"] = (
    storage_df["text_storage_bytes"] / 1024
).round(2)

storage_df["vector_storage_KB"] = (
    storage_df["estimated_vector_storage_bytes"] / 1024
).round(2)

storage_df[
    [
        "chunk_size",
        "num_chunks",
        "avg_chunk_chars",
        "text_storage_KB",
        "vector_storage_KB",
    ]
]

,chunk_size,num_chunks,avg_chunk_chars,text_storage_KB,vector_storage_KB
0,200,19,125.63,2.33,14.03
1,500,7,343.14,2.35,5.17
2,1000,3,803.33,2.35,2.21
3,2000,2,1206.00,2.36,1.48


In [23]:
def retrieve_chunks(query, chunks, top_k=2):
    texts = [chunk.page_content for chunk in chunks]

    vectorizer = TfidfVectorizer()

    document_vectors = vectorizer.fit_transform(texts)

    query_vector = vectorizer.transform([query])

    scores = (
        document_vectors @ query_vector.T
    ).toarray().flatten()

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "chunk": texts[index],
            "score": round(float(scores[index]), 4),
        })

    return results

In [24]:
# Test retrieval for every chunk size

retrieval_results = []

for size, chunks in chunk_experiment.items():

    question_scores = []

    for question in questions:

        results = retrieve_chunks(
            question,
            chunks,
            top_k=2,
        )

        best_score = results[0]["score"]

        question_scores.append(best_score)

    average_score = np.mean(question_scores)

    retrieval_results.append({
        "chunk_size": size,
        "retrieval_quality": round(
            float(average_score), 4
        ),
    })

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df

,chunk_size,retrieval_quality
0,200,0.3576
1,500,0.3957
2,1000,0.3148
3,2000,0.2432


In [25]:
# Display retrieved context for each question

for size in chunk_sizes:

    print("\n" + "=" * 80)
    print(f"CHUNK SIZE: {size}")
    print("=" * 80)

    for question in questions:

        results = retrieve_chunks(
            question,
            chunk_experiment[size],
            top_k=1,
        )

        print("\nQuestion:", question)
        print("Score:", results[0]["score"])
        print("Retrieved context:")
        print(results[0]["chunk"])


CHUNK SIZE: 200

Question: What is retrieval augmented generation?
Score: 0.5187
Retrieved context:
Retrieval-Augmented Generation, commonly called RAG, combines information
retrieval with language generation. A RAG system first retrieves relevant

Question: Why is chunk overlap useful?
Score: 0.2809
Retrieved context:
Chunk overlap helps preserve information when an important sentence or idea
crosses the boundary between two chunks. However, excessive overlap increases

Question: What is the purpose of metadata?
Score: 0.378
Retrieved context:
Metadata is also important in a retrieval system. Useful metadata includes
the source document, page number, section title, document type, and chunk ID.

Question: What are the problems with very small chunks?
Score: 0.292
Retrieved context:
When a user asks a question, the system searches for chunks that are
semantically related to the query.

Question: What are the problems with very large chunks?
Score: 0.3182
Retrieved context:
context. Ver

In [26]:
# Final Task 2 report

task2_report = storage_df.merge(
    retrieval_df,
    on="chunk_size",
)

task2_report["recommendation"] = ""

# Automatically assign a simple recommendation
best_size = task2_report.loc[
    task2_report["retrieval_quality"].idxmax(),
    "chunk_size"
]

for index, row in task2_report.iterrows():

    if row["chunk_size"] == best_size:
        task2_report.loc[
            index,
            "recommendation"
        ] = "Best retrieval score in this experiment"
    elif row["chunk_size"] <= 500:
        task2_report.loc[
            index,
            "recommendation"
        ] = "Good precision, but smaller context"
    else:
        task2_report.loc[
            index,
            "recommendation"
        ] = "More context, but potentially less precise"

task2_report[
    [
        "chunk_size",
        "num_chunks",
        "avg_chunk_chars",
        "retrieval_quality",
        "recommendation",
    ]
]

,chunk_size,num_chunks,avg_chunk_chars,retrieval_quality,recommendation
0,200,19,125.63,0.3576,"Good precision, but smaller context"
1,500,7,343.14,0.3957,Best retrieval score in this experiment
2,1000,3,803.33,0.3148,"More context, but potentially less precise"
3,2000,2,1206.00,0.2432,"More context, but potentially less precise"


## Task 2 Findings

The experiment demonstrates that chunk size affects:

- Number of chunks
- Vector/index size
- Amount of context returned
- Retrieval precision

### Important Observation

Smaller chunks create more retrievable units and can improve precision, but
they may not contain enough surrounding context.

Larger chunks reduce the number of chunks and provide more context, but the
retrieved text may contain more unrelated information.

### Recommendation

The final chunk size should be selected based on retrieval evaluation rather
than assuming that one value is universally optimal.

For many general RAG applications, a medium chunk size is a reasonable starting
point, followed by evaluation on real user questions.

> **Experiment limitation:** TF-IDF was used as a lightweight local retrieval
> proxy instead of a neural embedding model. Therefore, the retrieval scores
> should be interpreted as comparative experiment results, not production
> semantic-search benchmarks.

# Task 3: Smart Document Processor

The smart processor will:

1. Detect document type automatically.
2. Use `MarkdownHeaderTextSplitter` for Markdown.
3. Use `PythonCodeTextSplitter` for Python code.
4. Use `RecursiveCharacterTextSplitter` for plain text.
5. Preserve document structure.
6. Add intelligent overlap.
7. Add rich metadata.

### Metadata

Each chunk will contain:

- source
- document type
- section
- chunk ID
- token count
- overlap setting

In [27]:
from langchain_text_splitters import PythonCodeTextSplitter
import tiktoken
from pathlib import Path

In [28]:
# Token counter

encoding = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    return len(encoding.encode(text))

In [29]:
# Document type detector

def detect_document_type(source, content):
    extension = Path(source).suffix.lower()

    if extension == ".md":
        return "markdown"

    if extension == ".py":
        return "python"

    return "text"


print(detect_document_type("notes.md", markdown_document))
print(detect_document_type("model.py", "def train_model():"))
print(detect_document_type("article.txt", long_document))

markdown
python
text


In [30]:
def smart_document_processor(
    content,
    source,
    chunk_size=500,
):
    """
    Automatically detects document type and applies
    an appropriate LangChain splitter.
    """

    document_type = detect_document_type(
        source,
        content,
    )

    # Technical documents receive more overlap.
    if document_type in ["python", "markdown"]:
        overlap = int(chunk_size * 0.20)
    else:
        overlap = int(chunk_size * 0.10)

    # Select splitter
    if document_type == "markdown":

        headers = [
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
        ]

        header_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=headers,
            strip_headers=False,
        )

        sections = header_splitter.split_text(content)

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )

        chunks = splitter.split_documents(sections)

    elif document_type == "python":

        splitter = PythonCodeTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )

        chunks = splitter.create_documents([content])

    else:

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )

        chunks = splitter.create_documents([content])

    # Add rich metadata
    for index, chunk in enumerate(chunks, start=1):

        chunk.metadata["source"] = source
        chunk.metadata["type"] = document_type
        chunk.metadata["chunk_id"] = index
        chunk.metadata["tokens"] = count_tokens(
            chunk.page_content
        )
        chunk.metadata["overlap"] = overlap

        if "section" not in chunk.metadata:
            chunk.metadata["section"] = (
                chunk.metadata.get("Header 2")
                or chunk.metadata.get("Header 1")
                or "General"
            )

    return chunks

## 3A. Test with Markdown

In [33]:
markdown_chunks = smart_document_processor(
    content=markdown_document,
    source="day18_notes.md",
    chunk_size=300,
)

print("Document type: Markdown")
print("Chunks:", len(markdown_chunks))

for chunk in markdown_chunks:

    print("\n" + "=" * 70)
    print("Metadata:")
    print(chunk.metadata)

    print("\nContent:")
    print(chunk.page_content)

Document type: Markdown
Chunks: 7

Metadata:
{'Header 1': 'Artificial Intelligence', 'source': 'day18_notes.md', 'type': 'markdown', 'chunk_id': 1, 'tokens': 18, 'overlap': 60, 'section': 'Artificial Intelligence'}

Content:
# Artificial Intelligence  
Artificial intelligence enables computers to perform tasks associated with
human intelligence.

Metadata:
{'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'source': 'day18_notes.md', 'type': 'markdown', 'chunk_id': 2, 'tokens': 14, 'overlap': 60, 'section': 'Machine Learning'}

Content:
## Machine Learning  
Machine learning allows systems to learn patterns from data.

Metadata:
{'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Supervised Learning', 'source': 'day18_notes.md', 'type': 'markdown', 'chunk_id': 3, 'tokens': 16, 'overlap': 60, 'section': 'Machine Learning'}

Content:
### Supervised Learning  
Supervised learning uses labeled examples to train a model.

Metadata:
{'Hea

## 3B. Test with Python Code

In [34]:
python_code = '''
def clean_text(text):
    """Clean and normalize text."""
    text = text.strip()
    return text.lower()


def calculate_average(numbers):
    """Calculate the average of a list."""
    if not numbers:
        return 0

    return sum(numbers) / len(numbers)


def process_document(text):
    """Process a document using text cleaning."""
    cleaned = clean_text(text)
    return cleaned
'''

In [35]:
python_chunks = smart_document_processor(
    content=python_code,
    source="document_processor.py",
    chunk_size=300,
)

print("Document type: Python")
print("Chunks:", len(python_chunks))

for chunk in python_chunks:

    print("\n" + "=" * 70)
    print("Metadata:")
    print(chunk.metadata)

    print("\nCode:")
    print(chunk.page_content)

Document type: Python
Chunks: 2

Metadata:
{'source': 'document_processor.py', 'type': 'python', 'chunk_id': 1, 'tokens': 56, 'overlap': 60, 'section': 'General'}

Code:
def clean_text(text):
    """Clean and normalize text."""
    text = text.strip()
    return text.lower()


def calculate_average(numbers):
    """Calculate the average of a list."""
    if not numbers:
        return 0

    return sum(numbers) / len(numbers)

Metadata:
{'source': 'document_processor.py', 'type': 'python', 'chunk_id': 2, 'tokens': 24, 'overlap': 60, 'section': 'General'}

Code:
def process_document(text):
    """Process a document using text cleaning."""
    cleaned = clean_text(text)
    return cleaned


## 3C. Test with Plain Text

In [36]:
text_chunks = smart_document_processor(
    content=long_document,
    source="research_article.txt",
    chunk_size=300,
)

print("Document type: Plain Text")
print("Chunks:", len(text_chunks))

for chunk in text_chunks[:5]:

    print("\n" + "=" * 70)
    print("Metadata:")
    print(chunk.metadata)

    print("\nContent:")
    print(chunk.page_content)

Document type: Plain Text
Chunks: 10

Metadata:
{'source': 'research_article.txt', 'type': 'text', 'chunk_id': 1, 'tokens': 48, 'overlap': 30, 'section': 'General'}

Content:
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
language understanding, perception, and decision making.

Metadata:
{'source': 'research_article.txt', 'type': 'text', 'chunk_id': 2, 'tokens': 47, 'overlap': 30, 'section': 'General'}

Content:
Machine learning is a major branch of artificial intelligence. Machine
learning algorithms learn patterns from data instead of relying entirely on
manually written rules. Common machine learning approaches include supervised
learning, unsupervised learning, and reinforcement learning.

Metadata:
{'source': 'research_article.txt', 'type': 'text', 'chunk_id': 3, 'tokens': 44, 'overlap': 30, 'section': 'Gener

## 19. Smart Processor Results

### Document Type Detection

| Extension | Detected Type | Splitter |
|---|---|---|
| `.md` | Markdown | MarkdownHeaderTextSplitter + RecursiveCharacterTextSplitter |
| `.py` | Python | PythonCodeTextSplitter |
| `.txt` | Plain Text | RecursiveCharacterTextSplitter |

### Intelligent Overlap

Technical documents such as Markdown and Python receive more overlap because
code definitions, parameters, comments, and section relationships can depend
on surrounding context.

Narrative/plain text receives lower overlap to reduce duplication.

### Metadata

Every output chunk contains:

- Source
- Document type
- Chunk ID
- Token count
- Overlap
- Section/header information when available

# Day 18 Final Summary

## Task 1

`RecursiveCharacterTextSplitter` preserved context better than a simple fixed
character splitter because it considers natural separators and uses overlap.

## Task 2

Different chunk sizes produced different numbers of chunks and different
retrieval behavior.

Smaller chunks:
- More precise
- More chunks
- Less context

Larger chunks:
- More context
- Fewer chunks
- Potentially less precise

The experiment used TF-IDF as a local embedding/retrieval proxy because no paid
API or neural embedding model was required.

## Task 3

A smart document processor was implemented that automatically detects:

- Markdown
- Python
- Plain text

It then selects an appropriate splitter and preserves rich metadata.

## Overall Conclusion

Effective chunking is essential for RAG systems.

The goal is not simply to create small chunks. The goal is to create chunks that
are:

- Semantically coherent
- Retrieval-friendly
- Context-aware
- Efficient to store
- Traceable through metadata

A strong practical baseline is:

**Structure-aware splitting → recursive chunking → metadata preservation → embeddings → retrieval**